# 📊 Notebook 1 — Exploratory Data Analysis (EDA)
**Proyek:** Prediksi Potabilitas Air Sungai menggunakan Random Forest (PySpark)  
**Dataset:** `water_potability.csv` — 3.276 sampel, 10 fitur fisikokimia  
**Kelompok 1 — Tugas Besar Analisis Big Data**

---
## 1.1 Inisialisasi SparkSession

In [ ]:
from pyspark.sql import SparkSession
import pyspark

spark = SparkSession.builder \
    .appName('WaterPotability_EDA') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .config('spark.ui.port', '4040') \
    .getOrCreate()

print(f'PySpark Version : {pyspark.__version__}')
print(f'Spark UI        : http://localhost:4040')
print(f'SparkContext    : {spark.sparkContext}')

---
## 1.2 Membaca Dataset

In [ ]:
DATA_PATH = '/home/jovyan/work/data/water_potability.csv'

df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

print('=== SCHEMA ===')
df.printSchema()

print(f'\nJumlah baris : {df.count():,}')
print(f'Jumlah kolom : {len(df.columns)}')
print(f'Kolom        : {df.columns}')

In [ ]:
# Tampilkan 5 baris pertama
df.show(5, truncate=False)

---
## 1.3 Statistik Deskriptif

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

# Statistik deskriptif
stats = df.describe().toPandas()
stats = stats.set_index('summary').T
print('=== STATISTIK DESKRIPTIF ===')
stats

---
## 1.4 Identifikasi Missing Values

In [ ]:
from pyspark.sql.functions import col, isnan, when, count, round as spark_round

total_rows = df.count()

# Hitung missing values per kolom
missing_counts = df.select([
    count(when(col(c).isNull() | isnan(c), c)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

print('=== MISSING VALUES PER KOLOM ===')
print(f'{"Kolom":<25} {"Missing":>10} {"Persentase":>12}')
print('-' * 50)
for col_name, count_val in missing_counts.items():
    pct = (count_val / total_rows) * 100
    flag = ' ⚠️' if pct > 0 else ' ✅'
    print(f'{col_name:<25} {count_val:>10,} {pct:>11.1f}%{flag}')

---
## 1.5 Distribusi Label Target (Potability)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

label_dist = df.groupBy('Potability').count().orderBy('Potability').toPandas()
label_dist['Label'] = label_dist['Potability'].map({0: 'Tidak Layak (0)', 1: 'Layak (1)'})
label_dist['Persentase'] = (label_dist['count'] / total_rows * 100).round(1)

print('=== DISTRIBUSI LABEL ===')
print(label_dist[['Label', 'count', 'Persentase']].to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#e74c3c', '#27ae60']
axes[0].bar(label_dist['Label'], label_dist['count'], color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribusi Kelas Potabilitas', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel')
for i, (v, p) in enumerate(zip(label_dist['count'], label_dist['Persentase'])):
    axes[0].text(i, v + 10, f'{v:,}\n({p}%)', ha='center', fontsize=10)

axes[1].pie(label_dist['count'], labels=label_dist['Label'], autopct='%1.1f%%',
            colors=colors, startangle=90, shadow=True)
axes[1].set_title('Proporsi Kelas', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/home/jovyan/work/output/figures/distribusi_label.png', bbox_inches='tight')
plt.show()
print('✅ Gambar disimpan: output/figures/distribusi_label.png')

---
## 1.6 Heatmap Korelasi Antar Fitur

In [ ]:
import seaborn as sns
import numpy as np

# Konversi ke Pandas untuk visualisasi
df_pd = df.toPandas()

plt.figure(figsize=(12, 9))
corr_matrix = df_pd.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    annot_kws={'size': 9}
)
plt.title('Heatmap Korelasi — Water Potability Dataset', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/home/jovyan/work/output/figures/heatmap_korelasi.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Gambar disimpan: output/figures/heatmap_korelasi.png')

---
## 1.7 Density Plot Distribusi Setiap Fitur

In [ ]:
feature_cols = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
                'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

colors_by_class = {0: '#e74c3c', 1: '#27ae60'}
labels_by_class = {0: 'Tidak Layak (0)', 1: 'Layak (1)'}

for i, feat in enumerate(feature_cols):
    for cls in [0, 1]:
        subset = df_pd[df_pd['Potability'] == cls][feat].dropna()
        axes[i].hist(subset, bins=40, alpha=0.5, density=True,
                     color=colors_by_class[cls], label=labels_by_class[cls], edgecolor='none')
    axes[i].set_title(f'Distribusi: {feat}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=8)
    axes[i].grid(axis='y', alpha=0.3)

fig.suptitle('Distribusi Fitur Fisikokimia per Kelas Potabilitas', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/home/jovyan/work/output/figures/distribusi_fitur.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Gambar disimpan: output/figures/distribusi_fitur.png')

---
## 1.8 Ringkasan EDA

**Temuan Kunci:**
- Dataset memiliki **3.276 sampel** dengan **10 fitur fisikokimia**
- Distribusi label **tidak seimbang**: 61% tidak layak (0), 39% layak (1)
- **3 kolom** memiliki missing values: `ph` (491), `Sulfate` (781), `Trihalomethanes` (162)
- Korelasi antar fitur **lemah** — tidak ada fitur yang sangat dominan (tidak perlu PCA)
- Distribusi fitur menunjukkan **overlap** antara kedua kelas, sehingga model ensemble diperlukan

➡️ **Lanjut ke Notebook 02: Preprocessing**

In [ ]:
# Simpan DataFrame ke parquet untuk digunakan notebook berikutnya
df.write.mode('overwrite').parquet('/home/jovyan/work/output/df_raw.parquet')
print('✅ Raw DataFrame disimpan ke output/df_raw.parquet')

spark.stop()
print('SparkSession dihentikan.')